# TensorBoard with Fashion MNIST

<a target="_blank" href="https://colab.research.google.com/github/LuisAngelMendozaVelasco/luisangelmendozavelasco.github.io/blob/master/_portfolio/TensorFlow-Data_and_Deployment/portfolio-8.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png">Run in Google Colab</a>

**Objective:** Create a Convolutional Neural Network (CNN) to classify images of the Fashion MNIST dataset and use TensorBoard to explore the training process.

[TensorBoard](https://www.tensorflow.org/tensorboard) is a suite of web applications and visualization toolkit developed by the TensorFlow team to help users understand, debug, and optimize machine learning programs. Originally designed for TensorFlow, it is now widely used with other frameworks like PyTorch and Hugging Face Transformers to provide insights into the training process.

**Warning: This notebook is designed to be run in a Google Colab only.**

## Import libraries

In [1]:
# Load the TensorBoard notebook extension.
%load_ext tensorboard

In [2]:
import keras
from keras import Input, layers, Sequential
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import io

I0000 00:00:1785450696.700027  139584 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1785450696.735294  139584 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1785450697.463922  139584 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


## Load the dataset

We are going to use a CNN to classify images in the the [Fashion-MNIST dataset](https://github.com/zalandoresearch/fashion-mnist). This dataset consist of 70,000 grayscale images of fashion products from 10 categories, with 7,000 images per category. The images have a size of 28x28 pixels.

In [3]:
fashion_mnist = keras.datasets.fashion_mnist

(X_train, y_train), (X_right, y_right) = fashion_mnist.load_data()
X_validation, X_test, y_validation, y_test = train_test_split(X_right, y_right, train_size=0.5, random_state=0)
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

## Data preprocessing

Scale the pixel values to be between 0.0 and 1.0 and reshape the arrays to be 4-dimensional.

In [4]:
def preprocess_data(X):
    # Normalize the pixel values to be between 0 and 1
    X = X / 255.0
    # Reshape the data to add a channel dimension (for grayscale images)
    X = np.expand_dims(X, axis=-1)

    return X

In [5]:
X_train = preprocess_data(X_train)
X_test = preprocess_data(X_test)
X_validation = preprocess_data(X_validation)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")
print(f"X_validation shape: {X_validation.shape}, y_validation shape: {y_validation.shape}")

X_train shape: (60000, 28, 28, 1), y_train shape: (60000,)
X_test shape: (5000, 28, 28, 1), y_test shape: (5000,)
X_validation shape: (5000, 28, 28, 1), y_validation shape: (5000,)


## Visualize a single image

In [ ]:
# Clear out any prior log data.
!rm -rf /tmp/logs/train_data

# Sets up a timestamped log directory.
logdir = "/tmp/logs/train_data/" + datetime.now().strftime("%Y%m%d-%H%M%S")
# Creates a file writer for the log directory.
file_writer = tf.summary.create_file_writer(logdir)

# Using the file writer, log the reshaped image.
with file_writer.as_default():
    tf.summary.image("Training data", np.expand_dims(X_train[0], 0), step=0)

%tensorboard --logdir /tmp/logs/train_data

## Visualize an image generated by matplotlib

In [ ]:
# Clear out prior logging data.
!rm -rf /tmp/logs/plots

logdir = "/tmp/logs/plots/" + datetime.now().strftime("%Y%m%d-%H%M%S")
file_writer = tf.summary.create_file_writer(logdir)

def plot_to_image(figure):
    """Converts the matplotlib plot specified by 'figure' to a PNG image and
    returns it. The supplied figure is closed and inaccessible after this call."""
    # Save the plot to a PNG in memory.
    buf = io.BytesIO()
    plt.savefig(buf, format='png')
    # Closing the figure prevents it from being displayed directly inside
    # the notebook.
    plt.close(figure)
    buf.seek(0)
    # Convert PNG buffer to TF image
    image = tf.image.decode_png(buf.getvalue(), channels=4)
    # Add the batch dimension
    image = tf.expand_dims(image, 0)

    return image

def image_grid():
    """Return a 4x4 grid of the MNIST images as a matplotlib figure."""
    # Create a figure to contain the plot.
    indexes = np.random.choice(range(0, X_train.shape[0]), size=16, replace=False)
    samples = zip(X_train[indexes], y_train[indexes])

    fig, axs = plt.subplots(4, 4, figsize=(8, 8))
    fig.suptitle('Random samples')

    for ax, sample in zip(axs.flatten(), samples):
        ax.imshow(sample[0], cmap="gray")
        ax.set_title(class_names[sample[1]])
        ax.axis("off")

    plt.tight_layout()

    return fig

# Prepare the plot
figure = image_grid()

# Convert to image and log
with file_writer.as_default():
    tf.summary.image("Training data", plot_to_image(figure), step=0)

%tensorboard --logdir /tmp/logs/plots

## Build the CNN model

In [9]:
model = Sequential([
    Input(shape=(28, 28, 1)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2, 2),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D(2,2),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 11, 11, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1600)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 243,786 (952.29 KB)

 Trainable params: 243,786 (952.29 KB)

 Non-trainable params: 0 (0.00 B)

## Compile the model

In [10]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

## Train the model

Train the classifier and regularly log a confusion matrix along the way.

Here's what we'll do:
- Create the Keras TensorBoard callback to log basic metrics
- Create a Keras LambdaCallback to log the confusion matrix at the end of every epoch
- Train the model using Model.fit(), making sure to pass both callbacks

In [11]:
# Clear out prior logging data.
!rm -rf /tmp/logs/image

logdir = "/tmp/logs/image/" + datetime.now().strftime("%Y%m%d-%H%M%S")
# Define the basic TensorBoard callback.
tensorboard_callback = keras.callbacks.TensorBoard(log_dir=logdir)
file_writer_cm = tf.summary.create_file_writer(logdir + '/cm')

In [12]:
def log_confusion_matrix(epoch, logs):
    # Use the model to predict the values from the validation dataset.
    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    # Log the confusion matrix as an image summary.
    # figure = plot_confusion_matrix(y_test, y_pred)
    figure = ConfusionMatrixDisplay.from_predictions(y_test, y_pred).figure_
    cm_image = plot_to_image(figure)

    # Log the confusion matrix as an image summary.
    with file_writer_cm.as_default():
        tf.summary.image("epoch_confusion_matrix", cm_image, step=epoch)

# Define the per-epoch callback.
cm_callback = keras.callbacks.LambdaCallback(on_epoch_end=log_confusion_matrix)

In [ ]:
# Start TensorBoard.
%tensorboard --logdir /tmp/logs/image

# Train the classifier.
model.fit(
    X_train,
    y_train,
    epochs=5,
    verbose=0, # Suppress chatty output
    callbacks=[tensorboard_callback, cm_callback],
    validation_data=(X_validation, y_validation),
)